## Get the data

In [1]:
# shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-11 05:11:04--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-03-11 05:11:04 (22.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



## Load libraries and define necessary things

In [2]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

In [3]:
torch.manual_seed(2026)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
device

device(type='cuda')

## Load the data

In [4]:
# read the file
with open("input.txt") as f:
  text = f.read()

print(text[-1000:])

 while you take your rest,
And watch your safety.

ALONSO:
Thank you. Wondrous heavy.

SEBASTIAN:
What a strange drowsiness possesses them!

ANTONIO:
It is the quality o' the climate.

SEBASTIAN:
Why
Doth it not then our eyelids sink? I find not
Myself disposed to sleep.

ANTONIO:
Nor I; my spirits are nimble.
They fell together all, as by consent;
They dropp'd, as by a thunder-stroke. What might,
Worthy Sebastian? O, what might?--No more:--
And yet me thinks I see it in thy face,
What thou shouldst be: the occasion speaks thee, and
My strong imagination sees a crown
Dropping upon thy head.

SEBASTIAN:
What, art thou waking?

ANTONIO:
Do you not hear me speak?

SEBASTIAN:
I do; and surely
It is a sleepy language and thou speak'st
Out of thy sleep. What is it thou didst say?
This is a strange repose, to be asleep
With eyes wide open; standing, speaking, moving,
And yet so fast asleep.

ANTONIO:
Noble Sebastian,
Thou let'st thy fortune sleep--die, rather; wink'st
Whiles thou art waking.


In [5]:
# encode
vocab = sorted(set(text))
vocab_size = len(vocab)

print("Unique characters:", vocab)

Unique characters: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
# mapping to index value
char2idx = {char: i for i, char in enumerate(vocab)}
idx2char = {v: k for k, v in char2idx.items()}

In [7]:
encoded_text = torch.tensor([char2idx[char] for char in text])

In [8]:
# sequencing
seq_size = 100
step = 1
inputs = []
targets = []

for i in range(0, len(encoded_text) - seq_size - step + 1, step):
   inputs.append(encoded_text[i:i+seq_size])
   targets.append(encoded_text[i+step:i+seq_size+step])

inputs = torch.stack(inputs)
targets = torch.stack(targets)

In [9]:
data = TensorDataset(inputs, targets)

## Build **Transformer** Model

In [10]:
class Transformer(nn.Module):
    def __init__(self, *, vocab_size, emb_size, num_heads, hidden_size, max_seq_len=512):
        super().__init__()
        self.num_heads = num_heads
        self.hidden_size = hidden_size

        # text Embedding
        self.token_emb = nn.Embedding(vocab_size, emb_size)
        # positional Embedding
        self.pos_emb = nn.Embedding(max_seq_len, emb_size)

        # attention
        self.attn = nn.MultiheadAttention(
            emb_size,
            num_heads,
            batch_first=True,
            dropout=0.1
        )

        # normalize
        self.ln1 = nn.LayerNorm(emb_size)

        # dense layer
        self.ff = nn.Sequential(
            nn.Linear(emb_size, hidden_size * emb_size),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(hidden_size * emb_size, emb_size)
        )
        self.ln2 = nn.LayerNorm(emb_size)
        self.fc = nn.Linear(emb_size, vocab_size)

    def forward(self, x):
        B, T = x.shape # batch, sequence len
        pos = torch.arange(T, device=x.device) # position
        pos_emb = self.pos_emb(pos).unsqueeze(0) # positional embedding
        token_emb = self.token_emb(x) # token embedding
        x = pos_emb + token_emb

        # Causal mask
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        # attention
        residual = x # preserve the x
        x = self.ln1(x) # noramlize
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = attn_out + residual

        # feed forward
        residual = x
        x = self.ln2(x)
        ff_out = self.ff(x)
        x = residual + ff_out

        return self.fc(x)

In [11]:
model = Transformer(
  vocab_size=vocab_size,
  emb_size=256,
  hidden_size=128,
  num_heads=4,
  max_seq_len=seq_size
).to(device)

## Train the model

In [12]:
def train_model(model: torch.nn.Module, train_data: torch.utils.data.TensorDataset, batch: int=32, epochs: int=10, lr: float=0.01, verbose: int=0):
  if torch.cuda.device_count() > 1:
     model = nn.DataParallel(model)

  train_loader = DataLoader(data, batch_size=batch, pin_memory=True, shuffle=True)
  optimizer = torch.optim.Adam(model.parameters(), lr=lr)
  loss_fn = nn.CrossEntropyLoss()
  scheduler = torch.optim.lr_scheduler.OneCycleLR(
      optimizer,
      max_lr=lr,
      steps_per_epoch=len(train_loader),
      epochs=epochs
  )

  model.train()
  for epoch in range(epochs):
     total_loss = 0
     pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
     for i, (inp, tar) in enumerate(pbar):
        inp = inp.to(device, non_blocking=True)
        tar = tar.to(device, non_blocking=True)

        out = model(inp)
        loss = loss_fn(out.view(-1, vocab_size), tar.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()
        scheduler.step() # Step the scheduler after every batch

        current_loss = loss.item()
        total_loss += current_loss

        # Dynamically update the progress bar
        pbar.set_postfix({"Loss": f"{current_loss:.4f}", "Avg Loss": f"{total_loss/(i+1):.4f}", "LR": f"{scheduler.get_last_lr()[0]:.6f}"})

     if verbose == 1:
        print(f"Epoch {epoch+1} completed - Average Loss: {total_loss/len(train_loader):.4f}")

In [13]:
# Suggesting more epochs for better coherence
train_model(
    model,
    data,
    batch=256,
    epochs=2,
    lr=0.0006,
    verbose=1
)

Epoch 1/2:   0%|          | 0/4357 [00:00<?, ?it/s]

Epoch 1 completed - Average Loss: 1.6841


Epoch 2/2:   0%|          | 0/4357 [00:00<?, ?it/s]

Epoch 2 completed - Average Loss: 1.0740


## Generate Text

In [14]:
@torch.no_grad()
def generate_text(model, start_str, length=500, temperature=1.0):
    model.eval()
    chars = [char2idx[ch] for ch in start_str]
    # Start with the initial sequence
    input_ids = torch.tensor(chars).unsqueeze(0).to(device)
    generated = start_str

    for _ in range(length):
        # We only pass the last 'seq_size' characters to the model
        cond_input = input_ids[:, -seq_size:]

        logits = model(cond_input)
        # Focus on the last character's prediction
        logits = logits[:, -1, :] / temperature

        probs = torch.nn.functional.softmax(logits, dim=-1)
        next_char_idx = torch.multinomial(probs, num_samples=1)

        # Append the new character index to the running sequence
        input_ids = torch.cat((input_ids, next_char_idx), dim=1)

        generated += idx2char[next_char_idx.item()]

    return generated

In [15]:
print(generate_text(model, "ANTONIO:", 300))

ANTONIO:
'Twas a little in a lodged I bear my beauty.
This is the worst worse vastidity liege,
Tell me dead, blean ever been so fine crown.

RICHARD:
Now, Warwick now you both.

NORTHUMBERLAND:
Be it were I abhor, thanks, the black mourner,
Shall, for a while! Where's Marcius propagate,
And be it to thee, h


In [16]:
# To check relevance, let's look at the output again
print(generate_text(model, "Let's kill him", temperature=0.7))

Let's kill him, they daughter glister'd,
But say, I wake a quarrel friends a time call
The chopp'd our duke at Clarence take her from the wall.

GLOUCESTER:
Well, I have done to do him to the people
With Tybalt?

LUCENTIO:
Hath he will not rue can do the truth of his country,
How to lamentable and love in death.

DUCHESS OF YORK:
Why, there comes you wouldst thou wilt know,
What I can wish the king with thy stay we have
sent him from my son-in-law, and methinks,
To me comfort an any other cannot good to much 


In [17]:
torch.save(model.state_dict(), "Shakespeare_model.pth")